In [1]:
import os 
import json

def load_files(path, file_type='analysis'):
    audio_files = []
    
    for root, dirs, files in os.walk(path):
        for file in files:
            if file_type == 'normalized' and file.endswith("_normalized.json"):
                audio_files.append(os.path.join(root, file))
            elif file_type == 'analysis' and file.endswith("_analysis.json"):
                audio_files.append(os.path.join(root, file))
            elif file_type == 'vocab_corrected' and file.endswith("_vocab_corrected.json"):
                audio_files.append(os.path.join(root, file))
    return audio_files

In [2]:
# Define your collections and path
collections = ["lastfm", "suno", "udio"]
base_path = "../dataset_corrected/"
type = "normalized"

# Gather all jsons per collection
all_jsons = {}
for col in collections:
    dataset_path = f"{base_path}/{col}"
    all_jsons[col] = load_files(dataset_path, type)

# Optional: Print counts
for col in collections:
    print(f"{col}: {len(all_jsons[col])} JSON files")

lastfm: 19853 JSON files
suno: 19972 JSON files
udio: 19992 JSON files


In [3]:
# Sets for collecting unique chords
unique_chords = set()
unique_functional_chords = set()

# Loop over files and extract chords
for col, files in all_jsons.items():
    for file_path in files:
        with open(file_path, 'r') as f:
            data = json.load(f)
            for chord in data.get("chords", []):
                # Get chord name
                chord_name = chord.get("chord_name", "")
                if chord_name:
                    unique_chords.add(chord_name)
                
                # Get full functional harmony as a string or tuple
                fh = chord.get("functional_harmony", {})
                if fh:
                    functional_str = f"{fh.get('functional', '')}|{fh.get('alterations', '')}|{fh.get('roman_numeral', '')}"
                    unique_functional_chords.add(functional_str)

# Results
print(f"Total unique chord names: {len(unique_chords)}")
print(f"Total unique functional chords: {len(unique_functional_chords)}")

# Optional: list some of them
print("\nExamples of unique chord names:")
print(sorted(list(unique_chords))[:10])

print("\nExamples of unique functional chords:")
print(sorted(list(unique_functional_chords))[:10])

Total unique chord names: 392
Total unique functional chords: 798

Examples of unique chord names:
['A', 'A#', 'A#7', 'A#7#11', 'A#7#9', 'A#7alt', 'A#7b13', 'A#7b9', 'A#7sus', 'A#7sus4']

Examples of unique functional chords:
['#III|+|#III+', '#III|7532|#III7532', '#III|753b2|#III753b2', '#III|7|#III7', '#III|b75#43|#IIIb75#43', '#III|b753#2|#IIIb753#2', '#III|b7532|#IIIb7532', '#III|b753b2|#IIIb753b2', '#III|b753|#IIIb753', '#III|b7b653|#IIIb7b653']


In [4]:
import re

# Sets for vocabularies
root_vocabulary = set()
quality_vocabulary = set()

# Define root patterns manually (to ensure correct parsing)
all_roots = [
    "C", "C#", "Db", "D", "D#", "Eb", "E", "E#", "Fb", "F", "F#", "Gb",
    "G", "G#", "Ab", "A", "A#", "Bb", "B", "B#", "Cb"
]

# Sort roots by length descending to prioritize sharp/flat over base note
all_roots_sorted = sorted(all_roots, key=lambda x: -len(x))

for chord_name in unique_chords:
    root_found = False
    for root in all_roots_sorted:
        if chord_name.startswith(root):
            root_vocabulary.add(root)
            quality = chord_name[len(root):]
            quality_vocabulary.add(quality)
            root_found = True
            break
    if not root_found:
        print(f"Warning: Unrecognized root in chord name '{chord_name}'")

# Display results
print(f"\nTotal unique roots: {len(root_vocabulary)}")
print("Root vocabulary:", sorted(root_vocabulary))

print(f"\nTotal unique qualities: {len(quality_vocabulary)}")
print("Quality vocabulary:", sorted(quality_vocabulary))



Total unique roots: 19
Root vocabulary: ['A', 'A#', 'Ab', 'B', 'Bb', 'C', 'C#', 'Cb', 'D', 'D#', 'Db', 'E', 'Eb', 'F', 'F#', 'Fb', 'G', 'G#', 'Gb']

Total unique qualities: 23
Quality vocabulary: ['', '7', '7#11', '7#9', '7alt', '7b13', '7b9', '7sus', '7sus4', '9', 'aug', 'dim', 'dim7', 'm', 'm7', 'm7b5', 'm9', 'maj7', 'maj9', 'mmaj7', 'power', 'power9', 'sus4']
